In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="Lingalingeswaran/whisper-small-sinhala")

def transcribe(audio):
    # Transcribe the audio file to text
    print(f'transcribing file: {audio}...')
    text = pipe(audio, return_timestamps=True)["text"]
    print('transcription finished.')
    return text

In [ ]:
!pip install pyngrok

'''
from google.colab import userdata
ngrok_key = userdata.get('ngrok_key')
!ngrok config add-authtoken $ngrok_key'''

from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import os

saveDir = '/content/audiofiles'
try:
  os.mkdir(saveDir) # create save directory
except Exception as e:
  print(f'Dir already exists: {e}')

app = Flask(__name__)

@app.route('/',methods=['GET','POST'])
def receive():
  data = request.files['file']
  print(f'File Received: {data.filename}')

  data.save(f'{saveDir}/{data.filename}') # save to content/audiofiles

  text = transcribe(f'{saveDir}/{data.filename}') # extract trancription text
  print(text) 
  return jsonify({'reply':str(text)})

print(ngrok.connect(5000))
app.run(port=5000)